# Notes
Helpful info from udemy course: 
https://github.com/mrdbourke/pytorch-deep-learning/blob/main/01_pytorch_workflow.ipynb

# GitHub Repository:
https://github.com/mpennino/Future_DW_NO3

In [12]:
# Import libraries
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import random


import pyarrow as pa
import pyarrow.parquet as pq


In [13]:
# Make device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [14]:
# Load Observation Dataset
# DATA = readRDS(paste0(strap_dir,'Data/Models/RF_bi_model_All_DATA_all_vars_','Trends_Conc_PWS_GW_05to20', '.rds'))
#future_dir = 'C:/Users/MPennino/OneDrive - Environmental Protection Agency (EPA)/Projects/StRAPs/StRAP4/SSWR.405.1_Future_DW/Data/'
future_dir = 'C:/Users/MPennino/OneDrive - Environmental Protection Agency (EPA)/Projects/OASES/Data/Future_NO3/'


#dataset1 = 'Dataset_RF_Model_SW_COMIDS.csv'
dataset1 = 'Dataset_RF_Model_GW_COMIDS.csv'

input_data_ = pd.read_csv(future_dir+dataset1)
input_data_.head(3)

,COMID,viol_freq,PopDen2010Cat,PctCrop2019Cat,precip9120cat,tmean9120cat,HydrlCondCat,RockNCat,N_TW2012Cat,N_Surp_kgsqkm_2017cat,WtDepCat,ElevCat,Fe2O3Cat,SandCat,Hillslope_PctCat,Viol_Class
0,-504163,0,12.6213,36.43,1390.868070,20.671089,0.0557,83.7932,5.863261,5060.694776,121.2300,17.0535,0.7600,73.2634,0.859481,0
1,-504162,0,37.8686,11.85,1383.782884,20.638637,0.0671,89.8335,6.016214,5283.705054,121.2034,17.2614,0.7617,73.3484,0.938520,0
2,-504156,0,43.3216,27.39,1394.731896,20.595718,0.0557,84.6277,6.224132,6281.618355,159.5239,13.5562,0.7600,85.8590,0.895041,0


In [15]:
input_data1 = input_data_.drop(columns=['viol_freq']) 

names_list = input_data1.columns.tolist()
print(names_list), len(names_list) 

['COMID', 'PopDen2010Cat', 'PctCrop2019Cat', 'precip9120cat', 'tmean9120cat', 'HydrlCondCat', 'RockNCat', 'N_TW2012Cat', 'N_Surp_kgsqkm_2017cat', 'WtDepCat', 'ElevCat', 'Fe2O3Cat', 'SandCat', 'Hillslope_PctCat', 'Viol_Class']


(None, 15)

In [16]:
# Remove extra fields
# For Surface Water Dataset (,'AgDrain_pctWs','Hillslope_PctWs','BFIWs')
#input_data = input_data_.drop(columns=['HUC12','viol_freq','PopDen2010Ws','WaterInputWs','wdrw_LDWs','FertWs','CBNFWs','ManureWs','Septic_km2Cat']) 
input_data = input_data_.drop(columns=['COMID','viol_freq']) 

# For Groundwater Dataset
#input_data = input_data_.drop(columns=['HUC12','viol_freq','PopDen2010Cat','AgKffactCat','Septic_km2Cat','AgDrain_pctCat','WaterInputCat','wdrw_LDCat','BFICat','Hillslope_PctCat']) 

#input_data.head(3)
input_data.shape


(115843, 14)

In [17]:
# Calculate accuracy (a classification metric)
def accuracy_fn(y_true, y_pred):
    correct = torch.eq(y_true, y_pred).sum().item() # torch.eq() calculates where two tensors are equal
    acc = (correct / len(y_pred)) * 100 
    return acc

# Create Balanced Dataset


In [18]:
print(input_data['Viol_Class'].value_counts())

Viol_Class
0    114570
1      1273
Name: count, dtype: int64


In [19]:
# Save Preditor Data for SHAP Analysis
X_input_data = input_data.drop(columns=['Viol_Class']).values

# Convert to tensor data
X_input_data = torch.from_numpy(X_input_data).type(torch.float)

# get minimum size of the classes for balancing the dataset
min_size = input_data['Viol_Class'].value_counts().min()

X_input_data.shape,X_input_data.dtype, min_size


(torch.Size([115843, 13]), torch.float32, np.int64(1273))

In [20]:
# Find the size of the smallest class
min_size = input_data['Viol_Class'].value_counts().min()

min_size  = min_size * 10

# Sample exactly 'min_size' elements from each binary group
#balanced_df = input_data.groupby('Viol_Class').sample(n=min_size, random_state=42).reset_index(drop=True)

# If increasing the min_size, you can use the 'replace=True' argument to allow for sampling with replacement
balanced_df = input_data.groupby('Viol_Class').sample(n=min_size, random_state=42, replace=True).reset_index(drop=True)

print(balanced_df['Viol_Class'].value_counts())

Viol_Class
0    12730
1    12730
Name: count, dtype: int64


# Transform data to torch tensor


In [21]:
# Convert to tensors and split into train and test sets
from sklearn.model_selection import train_test_split
#X = input_data.drop(columns=['Viol_Class']).values # when use this the model just predicts the majority class
#y = input_data['Viol_Class'].values
X = balanced_df.drop(columns=['Viol_Class']).values
y = balanced_df['Viol_Class'].values

# Turn data into tensors
X = torch.from_numpy(X).type(torch.float)
y = torch.from_numpy(y).type(torch.float)

# Make a copy to use later
X_full = X.clone()
y_full = y.clone()

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y, 
                                                    test_size=0.2,
                                                    random_state=2
)

#X_train[:5], y_train[:5]

# Create NN Model

In [22]:
nrows = X_train.size()[0]
ncols = X_train.size()[1]
nrows,ncols

(20368, 13)

In [23]:
import torch
import torch.nn as nn

class ImprovedBinaryClassifier(nn.Module):
    def __init__(self, input_dim=ncols, hidden_dim=64):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LeakyReLU(0.1),             # Prevents dead neurons
            nn.BatchNorm1d(hidden_dim),     # Stabilizes training
            
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.2),                # Prevents overfitting
            
            nn.Linear(hidden_dim, 1)        # Outputs raw logits (No Sigmoid here - because signoid is used in next step when traning)
        )
        
    def forward(self, x):
        return self.network(x)

model1 = ImprovedBinaryClassifier().to(device)


# Train the Model

In [24]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, SubsetRandomSampler
from sklearn.model_selection import StratifiedKFold

# Mock data: 1000 samples, 10 features each
# X = np.random.randn(1000, 10).astype(np.float32)
# y = np.random.randint(0, 2, size=1000).astype(np.float32)

# # Convert to PyTorch Tensors
#X_tensor = torch.tensor(X)
#y_tensor = torch.tensor(y).unsqueeze(1) # Shape: [1000, 1] for BCELoss
X_tensor = X.clone().detach()
y_tensor = y.clone().detach().unsqueeze(1) # Shape: [1000, 1] for BCELoss
dataset = TensorDataset(X_tensor, y_tensor)


In [25]:
batchsize = int(len(dataset) * 0.2)
batchsize

5092

In [ ]:
# K-fold cross-validation model training
# SW: 3m for 500 epochs, 10 folds
# GW: 28 minutes for 500 epochs, 10 folds, 51 min for 1000 epochs, 10 folds
epochs = 1000
k_folds = 10 

seedvalue = 15 # 18 for SW
torch.manual_seed(seedvalue)

# Set the random seed for PyTorch (GPU / CUDA) if you use a graphics card
if torch.cuda.is_available():
  torch.cuda.manual_seed_all(seedvalue)

# Set the random seed for NumPy
np.random.seed(seedvalue)

# Set the random seed for Python's built-in random library
random.seed(seedvalue)

skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=seedvalue)

fold_results = []
models = []
val_samplers = [] # save test predictions for each fold for later analysis
val_loaders = [] # save test predictions for each fold for later analysis

# skf.split needs the original X and y to calculate class stratifications
for fold, (train_ids, val_ids) in enumerate(skf.split(X, y)): # val_ids are the indices for the validation (test) set for this fold
    print(f"--- FOLD {fold + 1} ---")
    
    # 1. Create data samplers for this specific fold
    train_sampler = SubsetRandomSampler(train_ids)
    val_sampler = SubsetRandomSampler(val_ids)
    
    # 2. Create DataLoaders
    train_loader = DataLoader(dataset, batch_size=batchsize, sampler=train_sampler, shuffle=False) # typical bastch_size = 32 ()
    val_loader = DataLoader(dataset, batch_size=batchsize, sampler=val_sampler, shuffle=False)
    
    # 3. INITIALIZE A FRESH MODEL AND OPTIMIZER FOR THIS FOLD
    model1 = ImprovedBinaryClassifier()
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model1.parameters(), lr=0.001)
    
    # 4. Training Loop for this fold
    model1.train()
    for epoch in range(epochs):
        for inputs, targets in train_loader:
            optimizer.zero_grad()
            outputs = model1(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
    # 5. Evaluation Loop for this fold
    model1.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in val_loader:
            outputs = model1(inputs)
            loss = criterion(outputs, targets)
            val_loss += loss.item() * inputs.size(0)
            
            # Convert logits to binary predictions (0 or 1)
            preds = (torch.sigmoid(outputs) >= 0.5).float()
            correct += (preds == targets).sum().item()
            total += targets.size(0)
            
    # Calculate metrics for this fold
    fold_acc = (correct / total) * 100
    fold_results.append(fold_acc) # store the accuracy for this fold
    models.append(model1) # store the trained model for this fold to use later for predictions or SHAP analysis
    val_samplers.append(val_sampler) # store the validation sampler for this fold
    val_loaders.append(val_loader) # store the validation/test dataset for this fold

    print(f"Fold {fold + 1} Accuracy: {fold_acc:.2f}%\n")

# Overall performance
print(f"Average {k_folds}-Fold Accuracy: {np.mean(fold_results):.2f}%")

--- FOLD 1 ---
Fold 1 Accuracy: 86.06%

--- FOLD 2 ---
Fold 2 Accuracy: 85.82%

--- FOLD 3 ---
Fold 3 Accuracy: 86.88%

--- FOLD 4 ---
Fold 4 Accuracy: 82.29%

--- FOLD 5 ---
Fold 5 Accuracy: 84.45%

--- FOLD 6 ---
Fold 6 Accuracy: 87.08%

--- FOLD 7 ---
Fold 7 Accuracy: 85.47%

--- FOLD 8 ---
Fold 8 Accuracy: 85.70%

--- FOLD 9 ---
Fold 9 Accuracy: 84.92%

--- FOLD 10 ---
Fold 10 Accuracy: 84.49%

Average 10-Fold Accuracy: 85.31%


In [215]:
#val_samplers[0].indices
#sampler_tensor = torch.tensor(val_samplers[0].indices)
#sampler_tensor.shape, sampler_tensor[0:5]

In [216]:
type(models), type(tests), type(models[0]), type(val_samplers[0]), type(X_test), type(y_test)

(list,
 list,
 __main__.ImprovedBinaryClassifier,
 torch.utils.data.sampler.SubsetRandomSampler,
 torch.Tensor,
 torch.Tensor)

# Model Evaluation Metrics
*PCC, Sensativity, Specificity, AUC

In [177]:
type(models[0]), len(models), len(val_loaders)

(__main__.ImprovedBinaryClassifier, 10, 10)

In [27]:
pcc_all = []
sensitivity_all = []
specificity_all = []
with torch.no_grad():
    
    for i in range(len(models)):
        model = models[i]
        model.eval()  # Set the model to evaluation mode
        val_loader = val_loaders[i]
        val_loss = 0.0
        correct = 0
        total = 0
        # Convert the dataloader into a Python iterator
        data_iterator = iter(val_loader)

        # Grab the first mini-batch
        inputs, targets = next(data_iterator)

        #for inputs, targets in val_loader:  # Using the ith validation loader
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        val_loss += loss.item() * inputs.size(0)
        
        # Convert logits to binary predictions (0 or 1)
        preds = (torch.sigmoid(outputs) >= 0.5).float()
        correct += (preds == targets).sum().item()
        total += targets.size(0)

        # Calculate True Positives, True Negatives, False Positives, False Negatives
        TP = torch.sum((preds == 1) & (targets == 1)).float()
        TN = torch.sum((preds == 0) & (targets == 0)).float()
        FP = torch.sum((preds == 1) & (targets == 0)).float()
        FN = torch.sum((preds == 0) & (targets == 1)).float()
        PCC = (TP + TN) / (TP + TN + FP + FN)
        sensitivity = TP / (TP + FN )
        specificity = TN / (TN + FP )
        #print(PCC, sensitivity, specificity)
        pcc_all.append(PCC)
        sensitivity_all.append(sensitivity)
        specificity_all.append(specificity)
        #print(len(preds), len(targets), len(inputs))

pcc_mean = np.mean(pcc_all)
sensitivity_mean = np.mean(sensitivity_all)
specificity_mean = np.mean(specificity_all)


#fold_acc = (sum(correct_all) / sum(total_all)) * 100
#fold_acc
#print(pcc_all)
print(f"PCC: {pcc_mean:.4f}")
print(f"Sensitivity (True Positives): {sensitivity_mean:.4f}")
print(f"Specificity (True Negatives): {specificity_mean:.4f}")

PCC: 0.8531
Sensitivity (True Positives): 0.8780
Specificity (True Negatives): 0.8283


In [218]:
len(pcc_all), len(sensitivity_all), len(specificity_all)

(10, 10, 10)

In [28]:
# AUC Calculation
from sklearn.metrics import roc_auc_score
import numpy as np

auc_scores = []
with torch.no_grad():
    for i in range(len(models)):
        model = models[i]
        model.eval()  # Set the model to evaluation mode
        val_loader = val_loaders[i]
        val_loss = 0.0
        correct = 0
        total = 0
        # Convert the dataloader into a Python iterator
        data_iterator = iter(val_loader)

        # Grab the first mini-batch
        inputs, targets = next(data_iterator)
        
        with torch.inference_mode():
            preds = torch.round(torch.sigmoid(model(inputs))).squeeze()

            preds2 = preds.detach().cpu().tolist()
            target2 = targets.detach().cpu().tolist()


            # 2. Calculate the AUC Score
            auc_score = roc_auc_score(target2, preds2)
            auc_scores.append(auc_score)

auc_mean = np.mean(auc_scores)
print(f"Test AUC: {auc_mean:.4f}")

Test AUC: 0.8531


# Save the Model

In [29]:
len(models)

10

In [30]:
model_dir = future_dir + "/Models"
model_dir

'C:/Users/MPennino/OneDrive - Environmental Protection Agency (EPA)/Projects/OASES/Data/Future_NO3//Models'

In [31]:
# Path
#modelpath = "/torch_models_5fold_future_NO3_sw.pth"
modelpath = "/torch_models_kfold_future_NO3_gw.pth"

# 1. Create a dictionary containing the state_dict of each model
models_checkpoint = {
    f"model_{i}": model.state_dict() for i, model in enumerate(models)
}

# 2. Save the dictionary to a single file (.pt or .pth extension)
torch.save(models_checkpoint, model_dir + modelpath)

# Save the test datasets

In [222]:
len(val_loaders)

10

In [32]:
import pickle

#testnames = "/test_datasets_sw.pkl"
testnames = "/test_datasets_gw.pkl"

# Extract state/parameters or pickle a list of datasets if serializable
dataset_list = [loader.dataset for loader in val_loaders]
len(dataset_list), type(dataset_list[0]), dataset_list[0].tensors[0].shape, dataset_list[0].tensors[1].shape

with open(model_dir + testnames, "wb") as f:
    pickle.dump(dataset_list, f)